# Ordered Logistic Regression Results Exploration with `mlcroissant`
This notebook guides you through loading and exploring the FAIR² dataset on predictors of knowledge adoption in rangeland management practices in Northern Kenya using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
meta = dataset.metadata  # Don't subsribe or iterate
print(meta.name + ': ' + meta.description)

## 2. Data Overview
Review available record sets, fields, and their IDs.

Each entity, such as record sets, fields, and columns, is referenced by its unique `@id`.

In [ ]:
# List all available record sets
record_sets = meta.record_sets
print('Available record sets:')
for rs in record_sets:
    print(f'@id: {rs.id}, name: {rs.name}, description: {rs.description}')

# For each record set, list fields and columns by @id
for rs in record_sets:
    print(f'\nRecord Set: {rs.name} (@id={rs.id})')
    if hasattr(rs, 'fields'):
        print(' Fields:')
        for f in rs.fields:
            print(f'   - {f.name} (@id={f.id}, dataType={getattr(f, "data_type", None)})')
    if hasattr(rs, 'columns'):
        print(' Columns:')
        for c in rs.columns:
            print(f'   - {c.name} (@id={c.id})')

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use only record set and field/column `@id`s.

Below, we list all record sets found and load their data.

In [ ]:
# Prepare to extract data using @id references
record_set_ids = [rs.id for rs in record_sets]
dfs = {}

for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dfs[rs_id] = df
        print(f'Loaded DataFrame for record set {rs_id}, columns: {df.columns.tolist()}')
        print(df.head(3))
    except Exception as e:
        print(f'Could not load {rs_id}: {e}')

# Select the primary record set (first one) for further EDA
main_record_set_id = record_set_ids[0]
df_main = dfs[main_record_set_id]
print(f'\nColumns for main record set (@id={main_record_set_id}):')
print(df_main.columns.tolist())
df_main.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering, normalizing, grouping. All field references used by their `@id`.

We'll select a numeric column, filter records, normalize the values, and optionally group by a categorical field (by @id).

In [ ]:
# Identify a numeric field (column) from the main record set
numeric_fields = [col for col in df_main.columns if df_main[col].dtype in ['int64', 'float64']]
if numeric_fields:
    numeric_field_id = numeric_fields[0]  # use @id as column name
    print(f'Using numeric field @id: {numeric_field_id}')
    threshold = 10
    filtered_df = df_main[df_main[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head(3))
    
    # Normalize
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, norm_col]].head())

    # Find possible group field (object or string)
    group_fields = [col for col in df_main.columns if df_main[col].dtype == 'object' and col != numeric_field_id]
    if group_fields:
        group_field_id = group_fields[0]
        print(f"Grouping by field @id: {group_field_id}")
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
        print(grouped_df.head())
else:
    print('No numeric field found for EDA.')

## 5. Visualization
Visualize data distributions and relationships for numeric fields by their `@id`.

Below, example visualizations for the main numeric and a categorical field:

In [ ]:
# Visualization imports
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_fields:
    # Histogram
    plt.figure(figsize=(6, 4))
    sns.histplot(df_main[numeric_field_id], kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    # Boxplot by group (if available)
    if group_fields:
        plt.figure(figsize=(8, 4))
        sns.boxplot(x=df_main[group_field_id], y=df_main[numeric_field_id])
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
This notebook demonstrated step-by-step usage of the `mlcroissant` library for exploring a FAIR² dataset, referencing all key entities by their `@id`.
- Loaded metadata and record sets from the Croissant schema.
- Enumerated fields and columns with `@id` for traceability.
- Performed basic EDA: filtering, normalization, grouping, and visualization.
- The dataset informs resilience and adaptation research, with possible limitations regarding bias and missing data.

Further exploration could focus on more sophisticated modeling, cross-record set joins, or deeper variable analysis using their `@id`.